In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
import os
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
import re
import nltk
nltk.download('punkt')

# Device selection: use CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == 'cuda':
    # clear cache and enable cuDNN benchmark for potential speedups
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    torch.backends.cudnn.benchmark = True

c:\Users\Bobby\Desktop\MasterThesis\Multilingual_Characterization\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Bobby\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
DATA_ROOT = "../data"
TRAIN_DATA_PARENT = os.path.join(DATA_ROOT, "target_4_December_release")
VAL_DATA_PARENT = os.path.join(DATA_ROOT, "cleaned_dev_10_january_2025")
TEST_DATA_PARENT = os.path.join(DATA_ROOT, "testdata_ST12")
TAXONOMY_FILE = os.path.join(DATA_ROOT, "taxonomy.json")

# Baseline-specific output paths (isolated from advanced model outputs)
BASELINE_CHECKPOINT_DIR = "../checkpoints/baseline_classifier"
BASELINE_PREDICTIONS_DIR = "../predictions/baseline"
BASELINE_DIAGRAMS_DIR = "../diagrams/baseline"

os.makedirs(BASELINE_PREDICTIONS_DIR, exist_ok=True)
os.makedirs(BASELINE_DIAGRAMS_DIR, exist_ok=True)

# Use validation set as test set (no separate test labels available)
TEST_DATA_PARENT = VAL_DATA_PARENT

In [3]:
# === Detect all available language folders ===
available_languages = [d for d in os.listdir(TRAIN_DATA_PARENT) if os.path.isdir(os.path.join(TRAIN_DATA_PARENT, d))]
print("Detected languages:", available_languages)

with open(TAXONOMY_FILE, "r", encoding="utf-8") as f:
    taxonomy = json.load(f)

label_data = []

for category in taxonomy:
    for subtype in category["subtypes"]:
        label_data.append({
            "main_category": category["name"],
            "subtype": subtype["name"],
            "description": subtype["description"],
            "example": subtype["example"]
        })

df = pd.DataFrame(label_data)
df.head()

Detected languages: ['BG', 'EN', 'HI', 'PT', 'RU']


,main_category,subtype,description,example
0,Protagonist,Guardian,A person who protects or defends something; or...,Police officers protecting citizens during a c...
1,Protagonist,Martyr,A person who is killed or who suffers greatly ...,Civil rights leaders like Martin Luther King J...
2,Protagonist,Peacemaker,"A person, group, or nation that tries to make ...",Nelson Mandela's efforts to reconcile South Af...
3,Protagonist,Rebel,A person who fights against the government of ...,Leaders of independence movements like Mahatma...
4,Protagonist,Underdog,"A person, team, or entity that is expected to ...",Grassroots political candidates overcoming wel...


In [4]:
main_categories = [cat["name"] for cat in taxonomy]
label2id = {label: i for i, label in enumerate(main_categories)}
id2label = {i: label for label, i in label2id.items()}

In [5]:
# === Dataset preparation ===
# === Dataset loader ===
def load_annotations(annotation_path, docs_root, labeled=True):
    data = []
    with open(annotation_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if labeled:
                # if len(parts) < 5:
                #     continue
                doc_id, mention, start, end, *labels = parts
                label = labels[0]
            else:
                # if len(parts) < 4:
                #     continue
                doc_id, mention, start, end = parts
                label = None

            start, end = int(start), int(end)
            text_path = os.path.join(docs_root, doc_id)
            # if not os.path.exists(text_path):
            #     continue

            with open(text_path, "r", encoding="utf-8") as doc_file:
                text = doc_file.read()

            entry = {
                "doc_id": doc_id,
                "text": text,
                "mention": mention,
                "start": start,
                "end": end,
            }
            if labeled:
                entry["label"] = label
            data.append(entry)
    return pd.DataFrame(data)

import nltk
nltk.download("punkt")

# --------------------------------------------------------------------
# Utility: given text, produce (span_start, span_end, substring)
# for paragraphs or sentences, maintaining global offsets.
# --------------------------------------------------------------------
def split_paragraphs_with_offsets(text):
    parts = []
    offset = 0
    for raw in text.split("\n"):
        p = raw.strip()
        if not p:
            offset += len(raw) + 1  # still move offset
            continue
        start = text.index(raw, offset)
        end = start + len(raw) - 1
        parts.append((start, end, raw))
        offset = end + 1
    return parts


def split_sentences_with_offsets(text):
    parts = []
    sentences = nltk.sent_tokenize(text)

    search_offset = 0
    for s in sentences:
        idx = text.find(s, search_offset)
        if idx == -1:
            idx = text.index(s)  # fallback
        parts.append((idx, idx + len(s), s))
        search_offset = idx + len(s)
    return parts


# --------------------------------------------------------------------
# Main expander: document → paragraphs or sentences
# This enforces correct mention span logic.
# --------------------------------------------------------------------
def expand_to_smaller_units(df, mode):
    new_rows = []

    for _, row in df.iterrows():
        text = row["text"]
        mention = row["mention"]
        orig_start = row["start"]
        orig_end = row["end"]

        # ----------------------------------------------------------------
        # choose unit splitter
        # ----------------------------------------------------------------
        if mode == "paragraph":
            units = split_paragraphs_with_offsets(text)
        elif mode == "sentence":
            units = split_sentences_with_offsets(text)
        else:
            raise ValueError("Unsupported mode.")

        # ----------------------------------------------------------------
        # Only keep the chunk that actually contains the original mention
        # ----------------------------------------------------------------
        for unit_start, unit_end, unit_text in units:

            # Does this unit include the absolute mention span?
            if not (unit_start <= orig_start < unit_end):
                continue

            # ----------------------------------------------------------------
            # Recalculate local start/end inside this chunk
            # ----------------------------------------------------------------
            mention_length = orig_end - orig_start + 1

            local_start = orig_start - unit_start
            local_end   = local_start + mention_length - 1


            # Additional safety check:
            if unit_text[local_start:local_end] != mention:
                # fallback: search within the chunk for exact location
                # This handles rare tokenization boundary issues
                idx = unit_text.find(mention)
                if idx == -1:
                    continue
                local_start = idx
                local_end = idx + len(mention) - 1

            # Build new row
            new_row = row.copy()
            new_row["text"] = unit_text

            # local offsets for the model
            new_row["start"] = local_start
            new_row["end"] = local_end

            # store ORIGINAL global offsets (critical!)
            new_row["orig_start"] = orig_start
            new_row["orig_end"] = orig_end

            new_rows.append(new_row)


    return pd.DataFrame(new_rows)


# === Combine data from all languages ===
def load_multilingual_data_by_mode(mode="document", labeled=True):

    assert mode in ["document", "paragraph", "sentence"]

    all_train, all_val, all_test = [], [], []

    for lang in available_languages:
        print(f"\n📘 Loading {mode}-level data for language: {lang}")

        train_root = os.path.join(TRAIN_DATA_PARENT, lang, "raw-documents")
        train_ann  = os.path.join(TRAIN_DATA_PARENT, lang, "subtask-1-annotations.txt")

        val_root = os.path.join(VAL_DATA_PARENT, lang, "subtask-1-documents")
        val_ann  = os.path.join(VAL_DATA_PARENT, lang, "subtask-1-annotations.txt")

        test_root = os.path.join(TEST_DATA_PARENT, lang, "subtask-1-documents")
        test_ann  = os.path.join(TEST_DATA_PARENT, lang, "subtask-1-annotations.txt")

        if not (os.path.exists(train_ann) and os.path.exists(val_ann)):
            print(f"⚠️ Skipping {lang}: missing annotation files")
            continue

        # ======================
        # Load document-level data
        # ======================
        train_df_doc = load_annotations(train_ann, train_root, labeled=labeled)
        val_df_doc   = load_annotations(val_ann, val_root, labeled=labeled)
        test_df_doc  = load_annotations(test_ann, test_root, labeled=labeled) if os.path.exists(test_ann) else pd.DataFrame()

        # ======================
        # Transform based on mode
        # ======================
        if mode == "document":
            train_df, val_df, test_df = train_df_doc, val_df_doc, test_df_doc

        else:
            train_df = expand_to_smaller_units(train_df_doc, mode)
            val_df   = expand_to_smaller_units(val_df_doc, mode)
            test_df  = expand_to_smaller_units(test_df_doc, mode) if not test_df_doc.empty else pd.DataFrame()

        # append
        all_train.append(train_df)
        all_val.append(val_df)
        all_test.append(test_df)

    return (
        pd.concat(all_train, ignore_index=True),
        pd.concat(all_val, ignore_index=True),
        pd.concat(all_test, ignore_index=True)
    )

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Bobby\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
# Load the data
train_df_full, val_df, test_df = load_multilingual_data_by_mode(mode="paragraph", labeled=True)

print(f"\n✅ Loaded dataset sizes: Train={len(train_df_full)}, Val={len(val_df)}, Test={len(test_df)}")

# Split the training data into new train and validation sets
train_df, new_val_df = train_test_split(train_df_full, test_size=0.2, random_state=42, stratify=train_df_full['label'])

# Use new validation set and keep using original validation set as test set
val_df = new_val_df
print(f"\n✅ Final dataset sizes: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

# Print class distribution
print("\n=== Label distribution ===")
print("Training set:")
print(train_df['label'].value_counts().to_string())
print("\nValidation set:")
print(val_df['label'].value_counts().to_string())
print("\nTest set:")
print(test_df['label'].value_counts().to_string())

train_df.head()


📘 Loading paragraph-level data for language: BG

📘 Loading paragraph-level data for language: EN

📘 Loading paragraph-level data for language: HI

📘 Loading paragraph-level data for language: PT

📘 Loading paragraph-level data for language: RU

✅ Loaded dataset sizes: Train=5606, Val=604, Test=604

✅ Final dataset sizes: Train=4484, Val=1122, Test=604

=== Label distribution ===
Training set:
label
Antagonist     2108
Protagonist    1491
Innocent        885

Validation set:
label
Antagonist     528
Protagonist    373
Innocent       221

Test set:
label
Antagonist     266
Protagonist    191
Innocent       147


,doc_id,text,mention,start,end,label,orig_start,orig_end
1865,HI_307.txt,मॉस्को: उत्तर कोरिया ने कहा कि वह रूस के साथ उ...,रूस,34,36,Protagonist,147,149
4315,PT_64.txt,A Procuradoria-Geral da Ucrânia afirma que os ...,Procuradoria-Geral da Ucrânia,2,30,Protagonist,1487,1515
769,EN_UA_026036.txt,KHARKIV - Russia pounded over 30 villages and...,Russia,11,16,Antagonist,77,82
3875,PT_27.txt,"""Repito mais uma vez sobre a existência do dec...",Zelensky,54,61,Antagonist,2289,2296
5080,RU-URW-1249.txt,"После ракетного удара по Киеву, в результате к...",Владимира Зеленского,362,381,Antagonist,362,381


In [7]:
# === Dataset class ===
class EntityFramingDataset(Dataset):
    def __init__(self, df, tokenizer, label2id, max_len=256, labeled=True):
        self.df = df
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_len = max_len
        self.labeled = labeled

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["text"]
        mention = row["mention"]
        marked_text = text.replace(mention, f"[ENTITY] {mention} [/ENTITY]")

        inputs = self.tokenizer(
            marked_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {key: val.squeeze(0) for key, val in inputs.items()}

        if self.labeled:
            label = self.label2id[row["label"]]
            item["labels"] = torch.tensor(label, dtype=torch.long)

        return item


In [8]:
# === Metrics ===
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="micro", zero_division=0
    )
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [9]:
# === Tokenizer and Model ===
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = EntityFramingDataset(train_df, tokenizer, label2id, labeled=True)
val_dataset = EntityFramingDataset(val_df, tokenizer, label2id, labeled=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Freeze all layers except the last transformer block, pooler, and classifier
for name, param in model.named_parameters():
    if not any(x in name for x in ['pooler', 'classifier', 'encoder.layer.11']):
        param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Percentage of trainable parameters: {100 * trainable_params / total_params:.2f}%')

model.to(device)

# === Training ===
training_args = TrainingArguments(
    output_dir=BASELINE_CHECKPOINT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

# === Evaluation ===
results = trainer.evaluate()
print("\n=== Validation Results ===")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 278,045,955
Trainable parameters: 7,680,771
Percentage of trainable parameters: 2.76%


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.929800,0.840912,0.626560,0.626560,0.626560,0.626560
2,0.924600,0.792070,0.656863,0.656863,0.656863,0.656863
3,0.983800,0.749542,0.688057,0.688057,0.688057,0.688057
4,0.679800,0.750515,0.691622,0.691622,0.691622,0.691622
5,0.689200,0.740408,0.686275,0.686275,0.686275,0.686275



=== Validation Results ===
eval_loss: 0.7505
eval_accuracy: 0.6916
eval_precision: 0.6916
eval_recall: 0.6916
eval_f1: 0.6916
eval_runtime: 57.0178
eval_samples_per_second: 19.6780
eval_steps_per_second: 2.4730
epoch: 5.0000


In [10]:
# === Generate and save predictions on test/val set ===
print("=== Generating predictions ===")

test_dataset = EntityFramingDataset(test_df, tokenizer, label2id, labeled=True)
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)
pred_labels = [id2label[i] for i in preds]

test_df = test_df.copy()
test_df["predicted_label"] = pred_labels

# Detect language from doc_id
# Doc IDs can be: "BG_123.txt", "EN_UA_026036.txt", "A9_BG_3970.txt",
# "A6_CC_BG_123.txt", "RU-URW-1249.txt", "HI_307.txt", etc.
def detect_language(doc_id):
    for lang in available_languages:
        # Match: starts with "LANG_" or "LANG-", or contains "_LANG_" / "_LANG-" / "-LANG-" / "-LANG_"
        if (doc_id.startswith(f"{lang}_") or doc_id.startswith(f"{lang}-")
                or f"_{lang}_" in doc_id or f"_{lang}-" in doc_id
                or f"-{lang}_" in doc_id or f"-{lang}-" in doc_id):
            return lang
    return "Unknown"

test_df["language"] = test_df["doc_id"].apply(detect_language)

# Verify language detection
lang_counts = test_df["language"].value_counts()
print(f"\nLanguage distribution:")
for lang, count in lang_counts.items():
    print(f"  {lang}: {count}")
if "Unknown" in lang_counts.index:
    unknown_ids = test_df[test_df["language"] == "Unknown"]["doc_id"].unique()
    print(f"\n⚠️ Unknown doc_ids: {unknown_ids[:5]}")

# Save predictions CSV
pred_path = os.path.join(BASELINE_PREDICTIONS_DIR, "baseline_predictions.csv")
test_df[["doc_id", "mention", "start", "end", "language", "label", "predicted_label"]].to_csv(
    pred_path, index=False
)
print(f"\nPredictions saved to: {pred_path}")
print(f"Total predictions: {len(test_df)}")
test_df.head()

=== Generating predictions ===

Language distribution:
  HI: 280
  PT: 116
  EN: 91
  RU: 86
  BG: 31

Predictions saved to: ../predictions/baseline\baseline_predictions.csv
Total predictions: 604


,doc_id,text,mention,start,end,label,orig_start,orig_end,predicted_label,language
0,A9_BG_3970.txt,"По-рано Орбан заяви, че Европа се втурва към в...",Европа,24,29,Antagonist,556,561,Antagonist,BG
1,A9_BG_3970.txt,"Той отбеляза, че изявленията на западните поли...",НАТО,160,163,Antagonist,1353,1356,Antagonist,BG
2,A9_BG_4076.txt,«Вашингтон предлага следното: «САЩ взема креди...,Европа,50,55,Innocent,440,445,Antagonist,BG
3,A9_BG_8427.txt,"""Обща цел за всички руснаци би трябвало да бъд...",Путин,94,98,Antagonist,1270,1274,Antagonist,BG
4,A9_BG_3819.txt,"""Живеем във време, когато Европа вече не е кон...",Украйна,233,239,Protagonist,492,498,Innocent,BG


In [11]:
# =====================================
# C1. Training Loss Curve
# =====================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import json
import glob

# Find the last checkpoint with trainer_state.json
ckpt_dirs = sorted(glob.glob(os.path.join(BASELINE_CHECKPOINT_DIR, "checkpoint-*")))
trainer_state_path = None
for d in reversed(ckpt_dirs):
    p = os.path.join(d, "trainer_state.json")
    if os.path.exists(p):
        trainer_state_path = p
        break

# Fallback: use trainer.state.log_history directly if available
if trainer_state_path:
    with open(trainer_state_path, "r") as f:
        state = json.load(f)
    log_history = state["log_history"]
else:
    log_history = trainer.state.log_history

train_steps = [e["step"] for e in log_history if "loss" in e]
train_loss = [e["loss"] for e in log_history if "loss" in e]
eval_steps = [e["step"] for e in log_history if "eval_loss" in e]
eval_loss = [e["eval_loss"] for e in log_history if "eval_loss" in e]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_steps, train_loss, label="Training Loss", alpha=0.7, linewidth=1.5)
if eval_loss:
    ax.plot(eval_steps, eval_loss, label="Validation Loss", marker='o', linewidth=2)
ax.set_xlabel("Training Step", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("Baseline Model: Training Loss Curve", fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(BASELINE_DIAGRAMS_DIR, "c1_baseline_training_loss.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: c1_baseline_training_loss.png")

Saved: c1_baseline_training_loss.png


C:\Users\Bobby\AppData\Local\Temp\ipykernel_18100\1532996162.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# =====================================
# C2. Confusion Matrix
# =====================================
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = test_df["label"].tolist()
y_pred = test_df["predicted_label"].tolist()
labels_order = ["Protagonist", "Antagonist", "Innocent"]

cm = confusion_matrix(y_true, y_pred, labels=labels_order)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_order)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Baseline Model: Confusion Matrix", fontsize=14, fontweight='bold')
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(BASELINE_DIAGRAMS_DIR, "c2_baseline_confusion_matrix.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: c2_baseline_confusion_matrix.png")

Saved: c2_baseline_confusion_matrix.png


C:\Users\Bobby\AppData\Local\Temp\ipykernel_18100\3953635376.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# =====================================
# C3. Per-Language F1
# =====================================

# Safety check: filter out any remaining "Unknown" language entries
test_df_lang = test_df[test_df["language"] != "Unknown"].copy()
if len(test_df_lang) < len(test_df):
    print(f"⚠️ Filtered out {len(test_df) - len(test_df_lang)} samples with unknown language")

lang_scores = []
for lang in sorted(test_df_lang["language"].unique()):
    mask = test_df_lang["language"] == lang
    yt = [label2id[l] for l in test_df_lang.loc[mask, "label"]]
    yp = [label2id[l] for l in test_df_lang.loc[mask, "predicted_label"]]
    acc = accuracy_score(yt, yp)
    _, _, f1, _ = precision_recall_fscore_support(yt, yp, average="weighted", zero_division=0)
    lang_scores.append({"Language": lang, "Accuracy": acc, "Weighted F1": f1, "Samples": mask.sum()})

lang_df = pd.DataFrame(lang_scores).sort_values("Weighted F1", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(lang_df["Language"], lang_df["Weighted F1"], color="#4C72B0", edgecolor="white")
for bar, val, n in zip(bars, lang_df["Weighted F1"], lang_df["Samples"]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.1%} (n={n})", va='center', fontsize=11)
ax.set_xlabel("Weighted F1", fontsize=12)
ax.set_title("Baseline Model: Weighted F1 per Language", fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.15)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(BASELINE_DIAGRAMS_DIR, "c3_baseline_per_language_f1.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: c3_baseline_per_language_f1.png")
print(lang_df.to_string(index=False))

Saved: c3_baseline_per_language_f1.png
Language  Accuracy  Weighted F1  Samples
      BG  0.645161     0.598750       31
      RU  0.627907     0.603784       86
      HI  0.632143     0.620541      280
      EN  0.714286     0.684760       91
      PT  0.827586     0.826375      116


C:\Users\Bobby\AppData\Local\Temp\ipykernel_18100\4288220877.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# =====================================
# C4. Per-Class Precision / Recall / F1
# =====================================

y_true_ids = [label2id[l] for l in test_df["label"]]
y_pred_ids = [label2id[l] for l in test_df["predicted_label"]]

class_prec, class_rec, class_f1, class_sup = precision_recall_fscore_support(
    y_true_ids, y_pred_ids, average=None, zero_division=0
)

classes = list(label2id.keys())
x = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width, class_prec, width, label="Precision", color="#4C72B0")
ax.bar(x,         class_rec,  width, label="Recall",    color="#DD8452")
ax.bar(x + width, class_f1,   width, label="F1",        color="#55A868")

for i in range(len(classes)):
    ax.text(x[i] - width, class_prec[i] + 0.02, f"{class_prec[i]:.2f}", ha='center', fontsize=9)
    ax.text(x[i],         class_rec[i]  + 0.02, f"{class_rec[i]:.2f}",  ha='center', fontsize=9)
    ax.text(x[i] + width, class_f1[i]   + 0.02, f"{class_f1[i]:.2f}",  ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(classes, fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Baseline Model: Per-Class Metrics", fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(BASELINE_DIAGRAMS_DIR, "c4_baseline_per_class_metrics.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: c4_baseline_per_class_metrics.png")

# Print summary
overall_acc = accuracy_score(y_true_ids, y_pred_ids)
_, _, overall_f1, _ = precision_recall_fscore_support(y_true_ids, y_pred_ids, average="weighted", zero_division=0)
print(f"\nOverall Accuracy: {overall_acc:.4f}")
print(f"Overall Weighted F1: {overall_f1:.4f}")
print(f"\nPer-class breakdown:")
for i, cls in enumerate(classes):
    print(f"  {cls}: P={class_prec[i]:.3f}  R={class_rec[i]:.3f}  F1={class_f1[i]:.3f}  Support={class_sup[i]}")

C:\Users\Bobby\AppData\Local\Temp\ipykernel_18100\222032789.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: c4_baseline_per_class_metrics.png

Overall Accuracy: 0.6821
Overall Weighted F1: 0.6802

Per-class breakdown:
  Protagonist: P=0.637  R=0.717  F1=0.675  Support=191
  Antagonist: P=0.662  R=0.744  F1=0.701  Support=266
  Innocent: P=0.856  R=0.524  F1=0.650  Support=147


# Fine-grained baseline classifier

Following the same baseline philosophy as the coarse classifier (XLM-RoBERTa-base + linear head + simple loss), we train an **independent** multi-label classifier over the 22 fine-grained roles. This classifier is intentionally not conditioned on the coarse predictions — it serves as a flat baseline that the advanced hierarchical system will be compared against.

Architecture / settings:
- `[CLS]` representation → linear head with 22 sigmoid outputs.
- **Loss**: `BCEWithLogitsLoss` with per-label `pos_weight = (#neg / #pos)` capped at 10. The vanilla unweighted BCE collapses to "predict nothing" because the trivial all-negative solution already gets a low loss when the average label has only ~5% positive rate. The cap of 10 (rather than the raw ratio, which can reach >100 for the rarest labels) prevents the opposite failure mode where the model floods every entity with positives — at cap=50 we observed mean predicted cardinality ≈ 7.7 vs true ≈ 1.1.
- **Threshold + fallback**: fixed 0.5, with argmax fallback when no label crosses the threshold (every entity has ≥1 fine label in this dataset, so an empty prediction is never correct).
- Same selective freezing as the coarse baseline (only layer 11 + pooler + classifier are trainable; ~2.77%).
- 10 epochs, batch 8, lr 3e-5, AdamW, weight_decay 0.01, max_len 256, paragraph granularity. (Compared to coarse: same batch size and weight decay; more epochs and slightly higher lr because the multi-label task is strictly harder than the 3-way coarse task.)

Metrics follow `src2/metrics.py` so baseline numbers are directly comparable to the advanced system: set-based Sample-F1, fine-only Exact Match Ratio (EMR = `gt_fine == pred_fine`), and the three hierarchical metrics (coarse accuracy, fine EMR, conditional fine F1 over correctly-classified coarse entities).

In [15]:
# =====================================
# F1. Constants & paths for fine baseline
# =====================================
BASELINE_FINE_CHECKPOINT_DIR = "../checkpoints/baseline_fine_classifier"
BASELINE_FINE_PREDICTIONS_DIR = "../predictions/baseline"
BASELINE_FINE_DIAGRAMS_DIR = "../diagrams/baseline"

os.makedirs(BASELINE_FINE_PREDICTIONS_DIR, exist_ok=True)
os.makedirs(BASELINE_FINE_DIAGRAMS_DIR, exist_ok=True)

# Build the flat list of 22 fine labels in the order they appear in the taxonomy
fine_labels = []
for category in taxonomy:
    for subtype in category["subtypes"]:
        fine_labels.append(subtype["name"])

fine_label2id = {lbl: i for i, lbl in enumerate(fine_labels)}
fine_id2label = {i: lbl for lbl, i in fine_label2id.items()}
NUM_FINE_LABELS = len(fine_labels)

# Map each fine label to its parent main role (used only for E2E analysis later)
fine_to_main = {}
for category in taxonomy:
    for subtype in category["subtypes"]:
        fine_to_main[subtype["name"]] = category["name"]

print(f"Fine labels ({NUM_FINE_LABELS}):")
for i, lbl in enumerate(fine_labels):
    print(f"  {i:2d}. {lbl:<20s}  (parent: {fine_to_main[lbl]})")

Fine labels (22):
   0. Guardian              (parent: Protagonist)
   1. Martyr                (parent: Protagonist)
   2. Peacemaker            (parent: Protagonist)
   3. Rebel                 (parent: Protagonist)
   4. Underdog              (parent: Protagonist)
   5. Virtuous              (parent: Protagonist)
   6. Instigator            (parent: Antagonist)
   7. Conspirator           (parent: Antagonist)
   8. Tyrant                (parent: Antagonist)
   9. Foreign Adversary     (parent: Antagonist)
  10. Traitor               (parent: Antagonist)
  11. Spy                   (parent: Antagonist)
  12. Saboteur              (parent: Antagonist)
  13. Corrupt               (parent: Antagonist)
  14. Incompetent           (parent: Antagonist)
  15. Terrorist             (parent: Antagonist)
  16. Deceiver              (parent: Antagonist)
  17. Bigot                 (parent: Antagonist)
  18. Forgotten             (parent: Innocent)
  19. Exploited             (parent: Innocent)


In [16]:
# =====================================
# F2. Multi-label data loading (fine roles)
# =====================================
def load_annotations_multilabel(annotation_path, docs_root):
    """Like load_annotations but keeps the FULL set of fine roles as a list."""
    data = []
    with open(annotation_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            # Format: doc_id, mention, start, end, main_role, fine_role_1, fine_role_2, ...
            if len(parts) < 5:
                continue
            doc_id, mention, start, end, main_role, *fine_roles = parts
            fine_roles = [r for r in fine_roles if r.strip()]
            start, end = int(start), int(end)
            text_path = os.path.join(docs_root, doc_id)

            with open(text_path, "r", encoding="utf-8") as doc_file:
                text = doc_file.read()

            data.append({
                "doc_id": doc_id,
                "text": text,
                "mention": mention,
                "start": start,
                "end": end,
                "label": main_role,
                "fine_labels": fine_roles,
            })
    return pd.DataFrame(data)


def load_multilingual_data_multilabel(mode="paragraph"):
    assert mode in ["document", "paragraph", "sentence"]

    all_train, all_val, all_test = [], [], []

    for lang in available_languages:
        print(f"📘 Loading {mode}-level multi-label data for language: {lang}")

        train_root = os.path.join(TRAIN_DATA_PARENT, lang, "raw-documents")
        train_ann  = os.path.join(TRAIN_DATA_PARENT, lang, "subtask-1-annotations.txt")

        val_root = os.path.join(VAL_DATA_PARENT, lang, "subtask-1-documents")
        val_ann  = os.path.join(VAL_DATA_PARENT, lang, "subtask-1-annotations.txt")

        test_root = os.path.join(TEST_DATA_PARENT, lang, "subtask-1-documents")
        test_ann  = os.path.join(TEST_DATA_PARENT, lang, "subtask-1-annotations.txt")

        if not (os.path.exists(train_ann) and os.path.exists(val_ann)):
            print(f"⚠️ Skipping {lang}: missing annotation files")
            continue

        train_df_doc = load_annotations_multilabel(train_ann, train_root)
        val_df_doc   = load_annotations_multilabel(val_ann, val_root)
        test_df_doc  = load_annotations_multilabel(test_ann, test_root) if os.path.exists(test_ann) else pd.DataFrame()

        if mode == "document":
            tr, va, te = train_df_doc, val_df_doc, test_df_doc
        else:
            tr = expand_to_smaller_units(train_df_doc, mode)
            va = expand_to_smaller_units(val_df_doc, mode)
            te = expand_to_smaller_units(test_df_doc, mode) if not test_df_doc.empty else pd.DataFrame()

        all_train.append(tr)
        all_val.append(va)
        all_test.append(te)

    return (
        pd.concat(all_train, ignore_index=True),
        pd.concat(all_val, ignore_index=True),
        pd.concat(all_test, ignore_index=True)
    )


# Load data with fine labels preserved
fine_train_df_full, fine_val_df, fine_test_df = load_multilingual_data_multilabel(mode="paragraph")
print(f"\n✅ Loaded multi-label dataset sizes: Train={len(fine_train_df_full)}, Val={len(fine_val_df)}, Test={len(fine_test_df)}")

# Same train/val split strategy as coarse baseline (stratified on main role to keep parity)
fine_train_df, fine_new_val_df = train_test_split(
    fine_train_df_full,
    test_size=0.2,
    random_state=42,
    stratify=fine_train_df_full["label"],
)
fine_val_df = fine_new_val_df
print(f"✅ Final multi-label sizes: Train={len(fine_train_df)}, Val={len(fine_val_df)}, Test={len(fine_test_df)}")

# Diagnostic: cardinality and label distribution in the train split
train_card = fine_train_df["fine_labels"].apply(len)
print(f"\nTrain fine-label cardinality: mean={train_card.mean():.2f}, median={train_card.median():.0f}, "
      f"min={train_card.min()}, max={train_card.max()}")

print("\nTrain fine-label support (sorted):")
from collections import Counter
train_counter = Counter(lbl for lbls in fine_train_df["fine_labels"] for lbl in lbls)
for lbl, cnt in sorted(train_counter.items(), key=lambda x: -x[1]):
    print(f"  {lbl:<20s} {cnt}")

📘 Loading paragraph-level multi-label data for language: BG
📘 Loading paragraph-level multi-label data for language: EN
📘 Loading paragraph-level multi-label data for language: HI
📘 Loading paragraph-level multi-label data for language: PT
📘 Loading paragraph-level multi-label data for language: RU

✅ Loaded multi-label dataset sizes: Train=5606, Val=604, Test=604
✅ Final multi-label sizes: Train=4484, Val=1122, Test=604

Train fine-label cardinality: mean=1.11, median=1, min=1, max=4

Train fine-label support (sorted):
  Victim               761
  Foreign Adversary    700
  Guardian             688
  Virtuous             414
  Instigator           317
  Incompetent          260
  Peacemaker           251
  Tyrant               218
  Rebel                189
  Conspirator          189
  Deceiver             183
  Terrorist            171
  Underdog             154
  Corrupt              133
  Exploited            97
  Saboteur             68
  Bigot                54
  Traitor         

In [17]:
# =====================================
# F3. Multi-label Dataset class
# =====================================
class EntityFramingFineDataset(Dataset):
    def __init__(self, df, tokenizer, fine_label2id, max_len=256, labeled=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.fine_label2id = fine_label2id
        self.num_labels = len(fine_label2id)
        self.max_len = max_len
        self.labeled = labeled

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["text"]
        mention = row["mention"]
        # Same string-replace marking as the coarse baseline
        marked_text = text.replace(mention, f"[ENTITY] {mention} [/ENTITY]")

        inputs = self.tokenizer(
            marked_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {key: val.squeeze(0) for key, val in inputs.items()}

        if self.labeled:
            multi_hot = torch.zeros(self.num_labels, dtype=torch.float)
            for lbl in row["fine_labels"]:
                if lbl in self.fine_label2id:
                    multi_hot[self.fine_label2id[lbl]] = 1.0
            item["labels"] = multi_hot

        return item

In [18]:
# =====================================
# F4. Metrics for multi-label classification
#
# Aligned with src2/metrics.py so results are directly comparable to the
# advanced system (sample-based set F1, fine-only EMR, etc.).
# =====================================
from sklearn.metrics import f1_score

# Standard 0.5 threshold. Combined with the moderate pos_weight cap below it keeps
# the predicted cardinality close to the true one (mean ≈ 1.1) instead of letting
# the model dump 7+ labels per entity.
FINE_THRESHOLD = 0.5


def _sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))


def _apply_threshold_with_fallback(probs, threshold):
    """Sigmoid probs → multi-hot predictions with argmax fallback.

    Every test entity has at least one fine label (mean true card ≈ 1.08),
    so empty predictions are never correct. If thresholding produces an
    all-zero row, we assign the single argmax label instead.
    """
    preds = (probs >= threshold).astype(int)
    empty_rows = preds.sum(axis=1) == 0
    if empty_rows.any():
        argmax_idx = probs[empty_rows].argmax(axis=1)
        fallback = np.zeros((empty_rows.sum(), probs.shape[1]), dtype=int)
        fallback[np.arange(empty_rows.sum()), argmax_idx] = 1
        preds[empty_rows] = fallback
    return preds


def _sample_f1(true_set, pred_set):
    """Set-based per-sample F1 — matches src2/metrics.py:compute_multilabel_f1_sample."""
    if len(true_set) == 0 and len(pred_set) == 0:
        return 1.0
    if len(pred_set) == 0 or len(true_set) == 0:
        return 0.0
    tp = len(true_set & pred_set)
    if tp == 0:
        return 0.0
    precision = tp / len(pred_set)
    recall = tp / len(true_set)
    return 2 * precision * recall / (precision + recall)


def _multihot_to_set(row, id2label):
    return {id2label[i] for i, v in enumerate(row) if v == 1}


def compute_metrics_multilabel(eval_pred):
    """Sigmoid + threshold + argmax fallback, then standard multi-label metrics."""
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    probs = _sigmoid_np(logits)
    preds = _apply_threshold_with_fallback(probs, FINE_THRESHOLD)
    labels = labels.astype(int)

    # Micro-F1 across the flattened binary matrix == standard multi-label micro-F1
    f1_micro = f1_score(labels.flatten(), preds.flatten(), average="binary", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)

    # Sample-F1 and EMR via set-based comparison (matches src2 semantics exactly)
    true_sets = [_multihot_to_set(r, fine_id2label) for r in labels]
    pred_sets = [_multihot_to_set(r, fine_id2label) for r in preds]
    f1_sample = float(np.mean([_sample_f1(t, p) for t, p in zip(true_sets, pred_sets)]))
    emr = float(np.mean([1.0 if t == p else 0.0 for t, p in zip(true_sets, pred_sets)]))

    mean_pred_card = float(preds.sum(axis=1).mean())
    mean_true_card = float(labels.sum(axis=1).mean())

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "f1_sample": f1_sample,
        "emr": emr,
        "mean_pred_card": mean_pred_card,
        "mean_true_card": mean_true_card,
    }

In [19]:
# =====================================
# F5. Train the fine baseline classifier
#
# Uses BCEWithLogitsLoss with pos_weight to counter the heavy class imbalance
# (~5% positive rate per label, mean cardinality ≈ 1.1 over 22 labels).
# Without pos_weight the model collapses to the trivial "predict nothing"
# minimum of BCE.
# =====================================
fine_tokenizer = AutoTokenizer.from_pretrained(model_name)

fine_train_dataset = EntityFramingFineDataset(fine_train_df, fine_tokenizer, fine_label2id, labeled=True)
fine_val_dataset   = EntityFramingFineDataset(fine_val_df,   fine_tokenizer, fine_label2id, labeled=True)

fine_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_FINE_LABELS,
    id2label=fine_id2label,
    label2id=fine_label2id,
    problem_type="multi_label_classification",
)

# Same selective freezing as the coarse baseline
for name, param in fine_model.named_parameters():
    if not any(x in name for x in ["pooler", "classifier", "encoder.layer.11"]):
        param.requires_grad = False

total_params = sum(p.numel() for p in fine_model.parameters())
trainable_params = sum(p.numel() for p in fine_model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Percentage of trainable parameters: {100 * trainable_params / total_params:.2f}%")

fine_model.to(device)

# --- pos_weight per fine label: (#negatives / #positives) on train split ---
# Standard BCEWithLogitsLoss imbalance correction: positive class is up-weighted
# so its gradient contribution roughly matches the negative class. We cap at 10
# (rather than the raw ratio, which can reach >100 for the rarest labels) to keep
# the model from over-predicting — at cap=50 the model dumped ~7 labels per entity.
n_train = len(fine_train_df)
pos_counts = np.zeros(NUM_FINE_LABELS, dtype=np.float64)
for lbls in fine_train_df["fine_labels"]:
    for lbl in lbls:
        if lbl in fine_label2id:
            pos_counts[fine_label2id[lbl]] += 1
neg_counts = n_train - pos_counts
# Safe ratio (avoid division by zero for absent labels)
pos_weight_np = np.where(pos_counts > 0, neg_counts / np.maximum(pos_counts, 1.0), 1.0)
# Moderate cap — strong enough to lift rare labels above the all-zero collapse,
# but not so strong that the model floods every prediction with positives.
pos_weight_np = np.clip(pos_weight_np, 1.0, 10.0)
pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32, device=device)

print("\npos_weight per label (capped at 10):")
for lbl, w in zip(fine_labels, pos_weight_np):
    print(f"  {lbl:<20s} pos={int(pos_counts[fine_label2id[lbl]]):4d}  weight={w:.2f}")


class WeightedBCETrainer(Trainer):
    """HF Trainer that uses BCEWithLogitsLoss with a per-label pos_weight."""

    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=self._pos_weight.to(logits.device))
        loss = loss_fct(logits, labels.float())
        return (loss, outputs) if return_outputs else loss


fine_training_args = TrainingArguments(
    output_dir=BASELINE_FINE_CHECKPOINT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,            # slightly higher than coarse (2e-5) — multi-label needs more updates
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,           # more epochs than coarse (5) — fine task is harder
    weight_decay=0.01,
    logging_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_sample",
    greater_is_better=True,
)

fine_trainer = WeightedBCETrainer(
    model=fine_model,
    args=fine_training_args,
    train_dataset=fine_train_dataset,
    eval_dataset=fine_val_dataset,
    processing_class=fine_tokenizer,
    compute_metrics=compute_metrics_multilabel,
    pos_weight=pos_weight,
)

fine_trainer.train()

fine_val_results = fine_trainer.evaluate()
print("\n=== Fine Baseline Validation Results ===")
for k, v in fine_val_results.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 278,060,566
Trainable parameters: 7,695,382
Percentage of trainable parameters: 2.77%

pos_weight per label (capped at 10):
  Guardian             pos= 688  weight=5.52
  Martyr               pos=  27  weight=10.00
  Peacemaker           pos= 251  weight=10.00
  Rebel                pos= 189  weight=10.00
  Underdog             pos= 154  weight=10.00
  Virtuous             pos= 414  weight=9.83
  Instigator           pos= 317  weight=10.00
  Conspirator          pos= 189  weight=10.00
  Tyrant               pos= 218  weight=10.00
  Foreign Adversary    pos= 700  weight=5.41
  Traitor              pos=  49  weight=10.00
  Spy                  pos=  16  weight=10.00
  Saboteur             pos=  68  weight=10.00
  Corrupt              pos= 133  weight=10.00
  Incompetent          pos= 260  weight=10.00
  Terrorist            pos= 171  weight=10.00
  Deceiver             pos= 183  weight=10.00
  Bigot                pos=  54  weight=10.00
  Forgotten            pos=  24  

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,F1 Sample,Emr,Mean Pred Card,Mean True Card
1,0.667000,0.669132,0.269489,0.117358,0.259418,0.042781,2.461676,1.116756
2,0.638700,0.615919,0.309686,0.163356,0.313307,0.047237,2.803030,1.116756
3,0.601800,0.590553,0.333793,0.205384,0.348067,0.081996,2.760250,1.116756
4,0.583700,0.574333,0.349441,0.207505,0.360027,0.084670,2.627451,1.116756
5,0.562400,0.564555,0.353999,0.215621,0.369466,0.087344,2.750446,1.116756
6,0.558300,0.549502,0.358036,0.227094,0.383767,0.095365,2.876114,1.116756
7,0.557000,0.543647,0.366089,0.242555,0.389269,0.089127,2.846702,1.116756
8,0.514100,0.540547,0.366885,0.239408,0.393396,0.105169,2.823529,1.116756
9,0.542500,0.533723,0.372820,0.264359,0.399234,0.098930,2.818182,1.116756
10,0.531700,0.533179,0.376183,0.265513,0.402676,0.107843,2.745098,1.116756



=== Fine Baseline Validation Results ===
eval_loss: 0.5332
eval_f1_micro: 0.3762
eval_f1_macro: 0.2655
eval_f1_sample: 0.4027
eval_emr: 0.1078
eval_mean_pred_card: 2.7451
eval_mean_true_card: 1.1168
eval_runtime: 78.5288
eval_samples_per_second: 14.2880
eval_steps_per_second: 1.7960
epoch: 10.0000


In [20]:
# =====================================
# F6. Predict on test set + save predictions
# =====================================
print("=== Generating fine-grained predictions ===")

fine_test_dataset = EntityFramingFineDataset(fine_test_df, fine_tokenizer, fine_label2id, labeled=True)
fine_predictions = fine_trainer.predict(fine_test_dataset)

fine_logits = fine_predictions.predictions
if isinstance(fine_logits, tuple):
    fine_logits = fine_logits[0]
fine_probs = _sigmoid_np(fine_logits)
fine_preds = _apply_threshold_with_fallback(fine_probs, FINE_THRESHOLD)

# Convert each row's multi-hot prediction to a list of label strings
def _row_to_labels(row_arr):
    return [fine_id2label[i] for i, v in enumerate(row_arr) if v == 1]

predicted_fine_label_lists = [_row_to_labels(row) for row in fine_preds]
true_fine_label_lists = [list(lbls) for lbls in fine_test_df["fine_labels"].tolist()]

fine_test_df = fine_test_df.copy()
fine_test_df["predicted_fine_labels"] = predicted_fine_label_lists
fine_test_df["true_fine_labels"] = true_fine_label_lists
fine_test_df["language"] = fine_test_df["doc_id"].apply(detect_language)

# Save predictions CSV
fine_pred_path = os.path.join(BASELINE_FINE_PREDICTIONS_DIR, "baseline_fine_predictions.csv")
fine_test_df[[
    "doc_id", "mention", "start", "end", "language",
    "label", "true_fine_labels", "predicted_fine_labels"
]].to_csv(fine_pred_path, index=False)
print(f"Fine predictions saved to: {fine_pred_path}")
print(f"Total predictions: {len(fine_test_df)}")

# Test set metrics (multi-label only — set-based, identical to src2/metrics.py)
y_true_fine = np.zeros((len(fine_test_df), NUM_FINE_LABELS), dtype=int)
for i, lbls in enumerate(true_fine_label_lists):
    for lbl in lbls:
        if lbl in fine_label2id:
            y_true_fine[i, fine_label2id[lbl]] = 1

true_sets_test = [set(lbls) for lbls in true_fine_label_lists]
pred_sets_test = [set(lbls) for lbls in predicted_fine_label_lists]

test_f1_micro  = f1_score(y_true_fine.flatten(), fine_preds.flatten(), average="binary", zero_division=0)
test_f1_macro  = f1_score(y_true_fine, fine_preds, average="macro", zero_division=0)
test_f1_sample = float(np.mean([_sample_f1(t, p) for t, p in zip(true_sets_test, pred_sets_test)]))
test_emr       = float(np.mean([1.0 if t == p else 0.0 for t, p in zip(true_sets_test, pred_sets_test)]))
test_mean_pred_card = float(fine_preds.sum(axis=1).mean())
test_mean_true_card = float(y_true_fine.sum(axis=1).mean())

print(f"\n=== Fine Baseline Test Results (multi-label only, threshold={FINE_THRESHOLD}) ===")
print(f"  Micro-F1:           {test_f1_micro:.4f}")
print(f"  Macro-F1:           {test_f1_macro:.4f}")
print(f"  Sample-F1:          {test_f1_sample:.4f}")
print(f"  Exact Match (EMR):  {test_emr:.4f}")
print(f"  Mean predicted card: {test_mean_pred_card:.2f}")
print(f"  Mean true card:      {test_mean_true_card:.2f}")

=== Generating fine-grained predictions ===
Fine predictions saved to: ../predictions/baseline\baseline_fine_predictions.csv
Total predictions: 604

=== Fine Baseline Test Results (multi-label only, threshold=0.5) ===
  Micro-F1:           0.3564
  Macro-F1:           0.2241
  Sample-F1:          0.3886
  Exact Match (EMR):  0.1175
  Mean predicted card: 2.70
  Mean true card:      1.08


In [21]:
# =====================================
# F7. Per-label F1 diagram (c5_baseline_fine_per_label_f1.png)
# =====================================
per_label_f1 = []
per_label_support = []
for j, lbl in enumerate(fine_labels):
    yt = y_true_fine[:, j]
    yp = fine_preds[:, j]
    f1_j = f1_score(yt, yp, zero_division=0)
    per_label_f1.append(f1_j)
    per_label_support.append(int(yt.sum()))

per_label_df = pd.DataFrame({
    "label": fine_labels,
    "main_role": [fine_to_main[lbl] for lbl in fine_labels],
    "f1": per_label_f1,
    "support": per_label_support,
})
per_label_df_sorted = per_label_df.sort_values("support", ascending=True).reset_index(drop=True)

# Color by parent main role for readability
role_colors = {"Protagonist": "#55A868", "Antagonist": "#C44E52", "Innocent": "#4C72B0"}
bar_colors = [role_colors[r] for r in per_label_df_sorted["main_role"]]

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(per_label_df_sorted["label"], per_label_df_sorted["f1"], color=bar_colors, edgecolor="white")
for i, (f1_v, sup_v) in enumerate(zip(per_label_df_sorted["f1"], per_label_df_sorted["support"])):
    ax.text(min(f1_v + 0.01, 0.98), i, f"{f1_v:.2f} (n={sup_v})", va="center", fontsize=8)
ax.set_xlim(0, 1.05)
ax.set_xlabel("F1", fontsize=12)
ax.set_title("Baseline fine — per-label F1 (sorted by test support)", fontsize=13, fontweight="bold")
ax.grid(axis="x", linestyle="--", alpha=0.4)

# Legend for main-role colors
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=c, label=r) for r, c in role_colors.items()]
ax.legend(handles=legend_handles, loc="lower right", fontsize=10)

plt.tight_layout()
per_label_path = os.path.join(BASELINE_FINE_DIAGRAMS_DIR, "c5_baseline_fine_per_label_f1.png")
fig.savefig(per_label_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {per_label_path}")
print(f"\nLabels with zero support in test set: {sum(1 for s in per_label_support if s == 0)}")
print(f"Labels with F1 == 0: {sum(1 for v in per_label_f1 if v == 0)}")
print(f"Mean per-label F1 (macro):       {np.mean(per_label_f1):.4f}")
nz_f1 = [f for f, s in zip(per_label_f1, per_label_support) if s > 0]
print(f"Mean F1 over labels with support: {np.mean(nz_f1):.4f}")

Saved: ../diagrams/baseline\c5_baseline_fine_per_label_f1.png

Labels with zero support in test set: 0
Labels with F1 == 0: 7
Mean per-label F1 (macro):       0.2241
Mean F1 over labels with support: 0.2241


C:\Users\Bobby\AppData\Local\Temp\ipykernel_18100\2327869917.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
# =====================================
# F8. Cardinality diagram (c6_baseline_fine_cardinality.png)
# =====================================
true_card = y_true_fine.sum(axis=1)
pred_card = fine_preds.sum(axis=1)

max_card = int(max(true_card.max(), pred_card.max()))
bins = np.arange(0, max_card + 2) - 0.5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of true vs predicted cardinality
ax = axes[0]
ax.hist(true_card, bins=bins, alpha=0.6, label=f"True (mean={true_card.mean():.2f})", color="#4C72B0", edgecolor="white")
ax.hist(pred_card, bins=bins, alpha=0.6, label=f"Predicted (mean={pred_card.mean():.2f})", color="#DD8452", edgecolor="white")
ax.set_xticks(np.arange(0, max_card + 1))
ax.set_xlabel("Number of fine labels per sample", fontsize=12)
ax.set_ylabel("Number of test samples", fontsize=12)
ax.set_title("Cardinality distribution: true vs predicted", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)

# Right: confusion-style 2D heatmap of (true_card, pred_card)
ax = axes[1]
heat = np.zeros((max_card + 1, max_card + 1), dtype=int)
for t, p in zip(true_card, pred_card):
    if 0 <= t <= max_card and 0 <= p <= max_card:
        heat[int(t), int(p)] += 1
sns.heatmap(heat, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=list(range(max_card + 1)), yticklabels=list(range(max_card + 1)),
            cbar=False)
ax.set_xlabel("Predicted cardinality", fontsize=12)
ax.set_ylabel("True cardinality", fontsize=12)
ax.set_title("True vs predicted cardinality (count of samples)", fontsize=13, fontweight="bold")

plt.tight_layout()
card_path = os.path.join(BASELINE_FINE_DIAGRAMS_DIR, "c6_baseline_fine_cardinality.png")
fig.savefig(card_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {card_path}")
print(f"  True cardinality:      mean={true_card.mean():.3f}, median={int(np.median(true_card))}, max={int(true_card.max())}")
print(f"  Predicted cardinality: mean={pred_card.mean():.3f}, median={int(np.median(pred_card))}, max={int(pred_card.max())}")
print(f"  Empty predictions (cardinality=0): {int((pred_card == 0).sum())} / {len(pred_card)}")
print(f"  Empty truths     (cardinality=0): {int((true_card == 0).sum())} / {len(true_card)}")

Saved: ../diagrams/baseline\c6_baseline_fine_cardinality.png
  True cardinality:      mean=1.084, median=1, max=3
  Predicted cardinality: mean=2.697, median=3, max=6
  Empty predictions (cardinality=0): 0 / 604
  Empty truths     (cardinality=0): 0 / 604


C:\Users\Bobby\AppData\Local\Temp\ipykernel_18100\2893394725.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
# =====================================
# F9. Per-language fine F1 (c7_baseline_fine_per_language_f1.png)
#
# Uses set-based Sample-F1 and EMR (matches src2/metrics.py).
# =====================================
fine_test_df_reset = fine_test_df.reset_index(drop=True)
lang_per_row = fine_test_df_reset["language"].tolist()

per_lang_rows = []
for lang in sorted(set(lang_per_row) - {"Unknown"}):
    idx = [i for i, l in enumerate(lang_per_row) if l == lang]
    if not idx:
        continue
    yt = y_true_fine[idx]
    yp = fine_preds[idx]
    t_sets = [true_sets_test[i] for i in idx]
    p_sets = [pred_sets_test[i] for i in idx]

    micro  = f1_score(yt.flatten(), yp.flatten(), average="binary", zero_division=0)
    macro  = f1_score(yt, yp, average="macro", zero_division=0)
    sample = float(np.mean([_sample_f1(t, p) for t, p in zip(t_sets, p_sets)]))
    emr_l  = float(np.mean([1.0 if t == p else 0.0 for t, p in zip(t_sets, p_sets)]))

    per_lang_rows.append({
        "Language": lang,
        "Samples": len(idx),
        "Micro-F1": micro,
        "Macro-F1": macro,
        "Sample-F1": sample,
        "EMR": emr_l,
    })

per_lang_df = pd.DataFrame(per_lang_rows).sort_values("Sample-F1", ascending=True).reset_index(drop=True)

x = np.arange(len(per_lang_df))
width = 0.28

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(x - width, per_lang_df["Micro-F1"],  width, label="Micro-F1",  color="#4C72B0")
ax.bar(x,         per_lang_df["Sample-F1"], width, label="Sample-F1", color="#DD8452")
ax.bar(x + width, per_lang_df["EMR"],       width, label="EMR",       color="#55A868")

for i in range(len(per_lang_df)):
    ax.text(x[i] - width, per_lang_df["Micro-F1"].iloc[i]  + 0.01, f"{per_lang_df['Micro-F1'].iloc[i]:.2f}",  ha="center", fontsize=9)
    ax.text(x[i],         per_lang_df["Sample-F1"].iloc[i] + 0.01, f"{per_lang_df['Sample-F1'].iloc[i]:.2f}", ha="center", fontsize=9)
    ax.text(x[i] + width, per_lang_df["EMR"].iloc[i]       + 0.01, f"{per_lang_df['EMR'].iloc[i]:.2f}",       ha="center", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([f"{r.Language}\n(n={r.Samples})" for r in per_lang_df.itertuples()], fontsize=11)
ax.set_ylabel("Score", fontsize=12)
ax.set_ylim(0, 1.0)
ax.set_title("Baseline fine — per-language metrics", fontsize=13, fontweight="bold")
ax.legend(fontsize=11, loc="upper left")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
lang_path = os.path.join(BASELINE_FINE_DIAGRAMS_DIR, "c7_baseline_fine_per_language_f1.png")
fig.savefig(lang_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {lang_path}")
print(per_lang_df.to_string(index=False))

Saved: ../diagrams/baseline\c7_baseline_fine_per_language_f1.png
Language  Samples  Micro-F1  Macro-F1  Sample-F1      EMR
      EN       91  0.223919  0.103445   0.215986 0.010989
      BG       31  0.268908  0.147710   0.251690 0.032258
      RU       86  0.346749  0.177185   0.345017 0.011628
      HI      280  0.344704  0.187396   0.360476 0.071429
      PT      116  0.591045  0.225867   0.660920 0.413793


C:\Users\Bobby\AppData\Local\Temp\ipykernel_18100\2031812256.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
# =====================================
# F10. Hierarchical baseline analysis (coarse + fine) + JSON metrics dump
#
# Three hierarchical metrics, identical to src2/metrics.py:evaluate_hierarchical_predictions:
#   1. coarse_acc            — coarse role correct
#   2. exact_match_acc (EMR) — fine label SET match (gt_fine == pred_fine)
#   3. conditional_fine_f1   — Sample-F1 over fine labels, restricted to entities
#                              where the coarse role was predicted correctly
# =====================================

# Align fine and coarse rows on (doc_id, mention, start, end). They were both produced
# from the same paragraph-expanded test set in identical order, so re-keying is just a safety net.
coarse_keyed = test_df.set_index(["doc_id", "mention", "start", "end"])["predicted_label"]
fine_keyed   = fine_test_df.set_index(["doc_id", "mention", "start", "end"])

assert len(coarse_keyed) == len(fine_keyed), \
    f"Coarse and fine test sizes diverge: {len(coarse_keyed)} vs {len(fine_keyed)}"

gt_coarse, pred_coarse = [], []
gt_fine_sets, pred_fine_sets = [], []
for key, frow in fine_keyed.iterrows():
    if key not in coarse_keyed.index:
        continue
    pc = coarse_keyed.loc[key]
    if isinstance(pc, pd.Series):
        pc = pc.iloc[0]
    gt_coarse.append(frow["label"])
    pred_coarse.append(pc)
    gt_fine_sets.append(set(frow["true_fine_labels"]))
    pred_fine_sets.append(set(frow["predicted_fine_labels"]))

n_total = len(gt_coarse)

# 1. Coarse accuracy
coarse_correct = sum(1 for gt, pr in zip(gt_coarse, pred_coarse) if gt == pr)
coarse_acc_e2e = coarse_correct / n_total

# 2. Coarse weighted-F1 (already computed earlier in c4 cell, reproduce here for the dump)
y_true_coarse_id = [label2id[l] for l in gt_coarse]
y_pred_coarse_id = [label2id[l] for l in pred_coarse]
_, _, coarse_wf1, _ = precision_recall_fscore_support(
    y_true_coarse_id, y_pred_coarse_id, average="weighted", zero_division=0
)

# 3. Exact match accuracy (EMR) — fine SET equality, per official SemEval-2025 ST1 definition
exact_matches = sum(1 for gt, pr in zip(gt_fine_sets, pred_fine_sets) if gt == pr)
exact_match_acc = exact_matches / n_total

# 4. Conditional fine F1 — Sample-F1 of fine labels, ONLY where coarse was correct
correct_idx = [i for i in range(n_total) if gt_coarse[i] == pred_coarse[i]]
if correct_idx:
    cond_f1_scores = [_sample_f1(gt_fine_sets[i], pred_fine_sets[i]) for i in correct_idx]
    conditional_fine_f1 = float(np.mean(cond_f1_scores))
else:
    conditional_fine_f1 = 0.0

print("\n=== Baseline hierarchical summary (independent coarse + fine classifiers) ===")
print(f"  Coarse Accuracy:           {coarse_acc_e2e:.4f}")
print(f"  Coarse Weighted-F1:        {coarse_wf1:.4f}")
print(f"  Fine Micro-F1:             {test_f1_micro:.4f}")
print(f"  Fine Macro-F1:             {test_f1_macro:.4f}")
print(f"  Fine Sample-F1:            {test_f1_sample:.4f}")
print(f"  Exact Match (fine, EMR):   {exact_match_acc:.4f}")
print(f"  Conditional Fine F1:       {conditional_fine_f1:.4f}  "
      f"(over {len(correct_idx)}/{n_total} entities with correct coarse)")

# --- Bar chart visualisation ---
fig, ax = plt.subplots(figsize=(11, 5.5))
metric_names = [
    "Coarse\nAccuracy",
    "Coarse\nWeighted-F1",
    "Fine\nMicro-F1",
    "Fine\nSample-F1",
    "Exact Match\n(EMR)",
    "Conditional\nFine F1",
]
metric_values = [coarse_acc_e2e, coarse_wf1, test_f1_micro, test_f1_sample,
                 exact_match_acc, conditional_fine_f1]
colors = ["#4C72B0", "#4C72B0", "#55A868", "#55A868", "#C44E52", "#8172B2"]

bars = ax.bar(metric_names, metric_values, color=colors, edgecolor="white")
for bar, v in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)
ax.set_ylim(0, max(max(metric_values) * 1.20, 0.1))
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Baseline — hierarchical metrics (coarse, fine, combined)", fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

e2e_path = os.path.join(BASELINE_FINE_DIAGRAMS_DIR, "c8_baseline_e2e_metrics.png")
fig.savefig(e2e_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {e2e_path}")

# --- JSON dump for LaTeX ingestion ---
metrics_dump = {
    "coarse": {
        "accuracy": coarse_acc_e2e,
        "weighted_f1": coarse_wf1,
    },
    "fine": {
        "f1_micro": test_f1_micro,
        "f1_macro": test_f1_macro,
        "f1_sample": test_f1_sample,
        "exact_match_acc": exact_match_acc,
        "mean_pred_card": test_mean_pred_card,
        "mean_true_card": test_mean_true_card,
        "threshold": FINE_THRESHOLD,
    },
    "hierarchical": {
        "coarse_accuracy": coarse_acc_e2e,
        "exact_match_accuracy": exact_match_acc,
        "conditional_fine_f1": conditional_fine_f1,
        "n_correct_coarse": len(correct_idx),
        "n_total": n_total,
    },
    "per_label_f1": {lbl: float(f) for lbl, f in zip(fine_labels, per_label_f1)},
    "per_label_support": {lbl: int(s) for lbl, s in zip(fine_labels, per_label_support)},
    "per_language": per_lang_df.to_dict(orient="records"),
    "n_test": int(len(fine_test_df)),
    "n_train": int(len(fine_train_df)),
    "n_val": int(len(fine_val_df)),
}

metrics_path = os.path.join(BASELINE_FINE_PREDICTIONS_DIR, "baseline_fine_metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_dump, f, indent=2, ensure_ascii=False)
print(f"Saved: {metrics_path}")

C:\Users\Bobby\AppData\Local\Temp\ipykernel_18100\2771892858.py:22: PerformanceWarning: indexing past lexsort depth may impact performance.
  if key not in coarse_keyed.index:
C:\Users\Bobby\AppData\Local\Temp\ipykernel_18100\2771892858.py:24: PerformanceWarning: indexing past lexsort depth may impact performance.
  pc = coarse_keyed.loc[key]



=== Baseline hierarchical summary (independent coarse + fine classifiers) ===
  Coarse Accuracy:           0.6854
  Coarse Weighted-F1:        0.6833
  Fine Micro-F1:             0.3564
  Fine Macro-F1:             0.2241
  Fine Sample-F1:            0.3886
  Exact Match (fine, EMR):   0.1175
  Conditional Fine F1:       0.4914  (over 414/604 entities with correct coarse)
Saved: ../diagrams/baseline\c8_baseline_e2e_metrics.png
Saved: ../predictions/baseline\baseline_fine_metrics.json


C:\Users\Bobby\AppData\Local\Temp\ipykernel_18100\2771892858.py:92: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Saliency Analysis — Word Contribution to Baseline Predictions

This section implements **occlusion saliency** and **gradient × embedding** attribution directly for the baseline classifiers (no dependency on `src2/`). For each token we measure how much the model's score for the target class changes when that token is masked — a positive delta means the word *supports* the predicted role, a negative delta means it *opposes* it.

Two methods are implemented:
1. **Occlusion saliency** (primary): replace each context token with `[MASK]`, measure delta in target logit.
2. **Gradient × Embedding** (secondary): dot product of ∇_{e_i} f_c and e_i — local linear approximation.

Both operate directly on the already-trained `model` (coarse) and `fine_model` (fine).


In [25]:
# =====================================
# S1. Saliency helper functions
#     (self-contained — no src2 dependency)
# =====================================
import textwrap

ENTITY_START_TOKEN = "[ENTITY]"
ENTITY_END_TOKEN   = "[/ENTITY]"

# ── token-level helpers ───────────────────────────────────────────────────────

def _saliency_special_ids(tok):
    """Collect IDs that must never be masked (specials + entity markers)."""
    ids = set()
    for tid in (tok.cls_token_id, tok.sep_token_id, tok.pad_token_id):
        if tid is not None:
            ids.add(int(tid))
    ids.add(int(tok.convert_tokens_to_ids(ENTITY_START_TOKEN)))
    ids.add(int(tok.convert_tokens_to_ids(ENTITY_END_TOKEN)))
    return ids


def _entity_span(input_ids_1d, ent_start_id, ent_end_id):
    starts = (input_ids_1d == ent_start_id).nonzero(as_tuple=True)[0]
    ends   = (input_ids_1d == ent_end_id  ).nonzero(as_tuple=True)[0]
    if len(starts) == 0 or len(ends) == 0:
        return None, None
    return int(starts[0]), int(ends[0])


def _valid_positions(input_ids_1d, attention_mask_1d, tok):
    """Token positions safe to occlude (not specials, not entity span)."""
    special = _saliency_special_ids(tok)
    ent_s_id = int(tok.convert_tokens_to_ids(ENTITY_START_TOKEN))
    ent_e_id = int(tok.convert_tokens_to_ids(ENTITY_END_TOKEN))
    s_pos, e_pos = _entity_span(input_ids_1d, ent_s_id, ent_e_id)
    positions = []
    for i in range(input_ids_1d.size(0)):
        if attention_mask_1d[i].item() != 1:
            continue
        tid = int(input_ids_1d[i].item())
        if tid in special:
            continue
        if s_pos is not None and s_pos <= i <= e_pos:
            continue
        positions.append(i)
    return positions


# ── forward pass scoring ──────────────────────────────────────────────────────

def _score(mdl, input_ids, attention_mask, target_idx, task, coarse_probs_t=None):
    """Return the raw target logit (shape (B,)) — unbounded, no saturation."""
    kwargs = {}
    if task == "fine" and coarse_probs_t is not None:
        # baseline fine model is AutoModelForSequenceClassification — no coarse_probs kwarg
        pass
    out = mdl(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
    return out.logits[:, target_idx]


# ── occlusion saliency ────────────────────────────────────────────────────────

def compute_occlusion_saliency_baseline(
    input_ids, attention_mask, mdl, target_idx, tok,
    task="coarse", batch_size=32, dev="cuda"
):
    """
    Batched occlusion saliency for baseline AutoModelForSequenceClassification.

    Returns
    -------
    saliency  : np.ndarray (T,)  — δ_i = f(x) - f(x_mask_i); 0 at skipped positions
    valid_mask: np.ndarray (T,) bool
    """
    input_ids      = input_ids.to(dev)
    attention_mask = attention_mask.to(dev)
    T = input_ids.size(1)
    sal   = np.zeros(T, dtype=np.float32)
    valid = np.zeros(T, dtype=bool)

    mdl.eval()
    with torch.no_grad():
        positions = _valid_positions(input_ids[0], attention_mask[0], tok)
        if not positions:
            return sal, valid

        p_orig = _score(mdl, input_ids, attention_mask, target_idx, task).item()
        mask_id = tok.mask_token_id
        n = len(positions)
        deltas = np.zeros(n, dtype=np.float32)
        pos_t  = torch.tensor(positions, dtype=torch.long, device=dev)

        for start in range(0, n, batch_size):
            end   = min(start + batch_size, n)
            chunk = end - start
            perturbed = input_ids.expand(chunk, -1).clone()
            am_rep    = attention_mask.expand(chunk, -1).contiguous()
            row_ids   = torch.arange(chunk, device=dev)
            col_ids   = pos_t[start:end]
            perturbed[row_ids, col_ids] = mask_id
            probs = _score(mdl, perturbed, am_rep, target_idx, task)
            deltas[start:end] = (p_orig - probs.detach().cpu().numpy()).astype(np.float32)

        for local_i, pos in enumerate(positions):
            sal[pos]   = deltas[local_i]
            valid[pos] = True

    return sal, valid


# ── gradient × embedding saliency ────────────────────────────────────────────

def compute_grad_x_emb_saliency_baseline(
    input_ids, attention_mask, mdl, target_idx, tok,
    task="coarse", dev="cuda"
):
    """
    Gradient × Embedding attribution for baseline AutoModelForSequenceClassification.

    Returns np.ndarray (T,) or None on failure.
    """
    input_ids      = input_ids.to(dev)
    attention_mask = attention_mask.to(dev)

    try:
        embed_module = mdl.roberta.embeddings.word_embeddings
    except AttributeError:
        return None

    cache = {}
    def _hook(module, inp, out):
        if isinstance(out, torch.Tensor):
            # word_embeddings weights are frozen (requires_grad=False) and the
            # inputs are integer IDs, so the embedding OUTPUT tensor would carry
            # requires_grad=False and retain_grad() would be a no-op (.grad stays
            # None). Force the output into the autograd graph so we can read its
            # gradient w.r.t. the target logit.
            if not out.requires_grad:
                out.requires_grad_(True)
            out.retain_grad()
            cache["emb"] = out
    handle = embed_module.register_forward_hook(_hook)

    was_train = mdl.training
    mdl.eval()
    try:
        with torch.enable_grad():
            mdl.zero_grad(set_to_none=True)
            out   = mdl(input_ids=input_ids, attention_mask=attention_mask)
            score = out.logits[0, target_idx]
            score.backward()
        emb = cache.get("emb")
        if emb is None or emb.grad is None:
            return None
        sal = (emb.grad * emb).sum(dim=-1).detach().cpu().numpy()[0]
    except Exception:
        return None
    finally:
        handle.remove()
        if was_train:
            mdl.train()

    # Zero out positions we don't want displayed
    invalid = np.ones_like(sal, dtype=bool)
    for pos in _valid_positions(input_ids[0], attention_mask[0], tok):
        invalid[pos] = False
    sal[invalid] = 0.0
    return sal.astype(np.float32)


# ── subword → word aggregation ────────────────────────────────────────────────

def aggregate_words(token_sal, encoding, marked_text, strategy="sum"):
    """
    Aggregate subword token saliency to whole-word level using word_ids().

    Returns list of dicts: {word, start, end, saliency}
    """
    word_ids = encoding.word_ids(batch_index=0)
    offsets  = encoding["offset_mapping"]
    if hasattr(offsets, "tolist"):
        offsets = offsets.tolist()
    if offsets and isinstance(offsets[0], (list, tuple)) and isinstance(offsets[0][0], (list, tuple)):
        offsets = offsets[0]

    words = {}
    for tok_i, wid in enumerate(word_ids):
        if wid is None:
            continue
        span = offsets[tok_i]
        s, e = int(span[0]), int(span[1])
        if s == 0 and e == 0:
            continue
        if wid not in words:
            words[wid] = {"start": s, "end": e, "vals": [float(token_sal[tok_i])]}
        else:
            words[wid]["start"] = min(words[wid]["start"], s)
            words[wid]["end"]   = max(words[wid]["end"],   e)
            words[wid]["vals"].append(float(token_sal[tok_i]))

    out = []
    for wid, w in words.items():
        text = marked_text[w["start"]:w["end"]]
        if text.strip() in (ENTITY_START_TOKEN, ENTITY_END_TOKEN):
            continue
        vals = w["vals"]
        if strategy == "max":
            sal = max(vals, key=abs)
        elif strategy == "mean":
            sal = float(np.mean(vals))
        else:
            sal = float(np.sum(vals))
        out.append({"word": text, "start": w["start"], "end": w["end"], "saliency": sal})
    out.sort(key=lambda x: x["start"])
    return out


# ── bar chart ─────────────────────────────────────────────────────────────────

def plot_saliency_bar_baseline(words, target_label, top_k=10, title=None):
    """
    Horizontal bar chart: top-K words by |saliency|, bars coloured by sign.
    Each bar is labelled with signed share (%) of total absolute influence.
    """
    if not words:
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.text(0.5, 0.5, "No context", ha="center", va="center")
        ax.axis("off")
        return fig

    total_abs = max(sum(abs(w["saliency"]) for w in words), 1e-12)
    ranked = sorted(words, key=lambda w: abs(w["saliency"]), reverse=True)[:top_k]
    ranked.sort(key=lambda w: w["saliency"], reverse=True)

    labels = ["\n".join(textwrap.wrap(w["word"], 22)) or w["word"] for w in ranked]
    shares = [w["saliency"] / total_abs * 100.0 for w in ranked]
    colors = ["#2E7D32" if s > 0 else "#C62828" for s in shares]

    max_abs = max((abs(s) for s in shares), default=1.0)
    k = len(ranked)
    fig, ax = plt.subplots(figsize=(6.5, max(2.5, 0.5 * k)))
    y_pos = np.arange(k)
    bars  = ax.barh(y_pos, shares, color=colors, edgecolor="white", height=0.7)
    off   = max_abs * 0.03
    for bar, s in zip(bars, shares):
        x  = bar.get_width()
        ha = "left" if x >= 0 else "right"
        ax.text(x + (off if x >= 0 else -off), bar.get_y() + bar.get_height() / 2,
                f"{s:+.1f}%", va="center", ha=ha, fontsize=9)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=10)
    ax.invert_yaxis()
    ax.axvline(0, color="#888", linewidth=0.8)
    ax.set_xlabel(f"Share of total influence on {target_label} (%)", fontsize=9)
    if title:
        ax.set_title(title, fontsize=11, fontweight="bold")
    ax.grid(axis="x", alpha=0.25, linewidth=0.5)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    xmin = min(shares + [0.0])
    xmax = max(shares + [0.0])
    span = (xmax - xmin) if xmax > xmin else max(max_abs, 1.0)
    ax.set_xlim(xmin - span * 0.30, xmax + span * 0.30)
    fig.tight_layout()
    return fig


# ── tokenization helper ───────────────────────────────────────────────────────

def _tokenize_marked(marked_text, tok, max_len=256):
    """Return encoding with offset_mapping (requires fast tokenizer)."""
    return tok(
        marked_text,
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt",
        return_offsets_mapping=True,
    )


print("Saliency helpers loaded.")


Saliency helpers loaded.


In [26]:
# =====================================
# S2. Occlusion saliency on selected examples — coarse + fine classifiers
#     Saves diagrams c9_baseline_saliency_examples.png
# =====================================
import time

dev_str = "cuda" if torch.cuda.is_available() else "cpu"

# Three representative test examples spanning the three coarse classes. Correctness
# (✓/✗) is determined dynamically from the model prediction, not hard-coded.
SALIENCY_EXAMPLES = [
    {
        "label": "Пример 1 (истинско: Antagonist)",
        "text": (
            "KHARKIV - Russia pounded over 30 villages and towns in Ukraine's northeastern "
            "Kharkiv region on Tuesday, killing at least one civilian and injuring six, "
            "regional governor Oleh Syniehubov said."
        ),
        "mention": "Russia",
        "true_coarse": "Antagonist",
    },
    {
        "label": "Пример 2 (истинско: Innocent)",
        "text": (
            'Западът отгледа "терористична гадина", която унищожава всичко. '
            "Така официалният представител на МВнР на Русия Мария Захарова коментира "
            "пред журналисти удара по Луганск."
        ),
        "mention": "Луганск",
        "true_coarse": "Innocent",
    },
    {
        "label": "Пример 3 (истинско: Protagonist)",
        "text": (
            "Президентът на Украйна Володимир Зеленски призова за пълна защита на украинското "
            "небе след масираната руска въздушна атака, предаде Укринформ."
        ),
        "mention": "Володимир Зеленски",
        "true_coarse": "Protagonist",
    },
]

def run_example_saliency(ex, mdl, fine_mdl, tok, fine_tok, dev, top_k=8):
    """Run occlusion saliency for coarse + fine classifier on one example."""
    mention = ex["mention"]
    text    = ex["text"]
    # string-replace marking (same as baseline training)
    marked  = text.replace(mention, f"{ENTITY_START_TOKEN} {mention} {ENTITY_END_TOKEN}")

    enc = _tokenize_marked(marked, tok)
    inp = enc["input_ids"].to(dev)
    am  = enc["attention_mask"].to(dev)

    # ── coarse prediction ─────────────────────────────────────────────────────
    mdl.eval()
    with torch.no_grad():
        coarse_logits = mdl(input_ids=inp, attention_mask=am).logits[0]
        coarse_probs  = torch.softmax(coarse_logits, dim=-1).cpu().numpy()
        pred_coarse_id = int(coarse_probs.argmax())
        pred_coarse    = id2label[pred_coarse_id]

    # ── fine prediction ───────────────────────────────────────────────────────
    fine_enc = _tokenize_marked(marked, fine_tok)
    fine_inp = fine_enc["input_ids"].to(dev)
    fine_am  = fine_enc["attention_mask"].to(dev)

    fine_mdl.eval()
    with torch.no_grad():
        fine_logits = fine_mdl(input_ids=fine_inp, attention_mask=fine_am).logits[0]
        fine_probs  = torch.sigmoid(fine_logits).cpu().numpy()
        fine_preds_bin = _apply_threshold_with_fallback(fine_probs[None], FINE_THRESHOLD)[0]
        pred_fine_labels = [fine_id2label[j] for j, v in enumerate(fine_preds_bin) if v == 1]

    # ── coarse saliency ───────────────────────────────────────────────────────
    t0 = time.perf_counter()
    sal_coarse, _ = compute_occlusion_saliency_baseline(
        inp, am, mdl, pred_coarse_id, tok, task="coarse", dev=dev
    )
    words_coarse = aggregate_words(sal_coarse, enc, marked)
    elapsed_c = (time.perf_counter() - t0) * 1000

    # ── fine saliency (for top predicted fine label) ──────────────────────────
    if pred_fine_labels:
        top_fine_lbl = pred_fine_labels[0]
        top_fine_id  = fine_label2id[top_fine_lbl]
        t1 = time.perf_counter()
        sal_fine, _ = compute_occlusion_saliency_baseline(
            fine_inp, fine_am, fine_mdl, top_fine_id, fine_tok, task="fine", dev=dev
        )
        words_fine = aggregate_words(sal_fine, fine_enc, marked)
        elapsed_f  = (time.perf_counter() - t1) * 1000
    else:
        top_fine_lbl = None
        words_fine   = []
        elapsed_f    = 0.0

    return {
        "marked": marked,
        "pred_coarse": pred_coarse,
        "pred_fine": pred_fine_labels,
        "coarse_probs": {id2label[i]: float(coarse_probs[i]) for i in range(3)},
        "words_coarse": words_coarse,
        "words_fine": words_fine,
        "top_fine_lbl": top_fine_lbl,
        "elapsed_c_ms": elapsed_c,
        "elapsed_f_ms": elapsed_f,
    }


# Run saliency for all three examples
print("Running occlusion saliency on 3 examples...")
results_sal = []
for ex in SALIENCY_EXAMPLES:
    print(f"  → {ex['label']} ...", end=" ", flush=True)
    r = run_example_saliency(ex, model, fine_model, tokenizer, fine_tokenizer,
                              dev_str, top_k=8)
    results_sal.append(r)
    print(f"coarse={r['pred_coarse']}  fine={r['pred_fine']}  "
          f"({r['elapsed_c_ms']:.0f}ms + {r['elapsed_f_ms']:.0f}ms)")


Running occlusion saliency on 3 examples...
  → Antagonist (correct) ... coarse=Protagonist  fine=['Guardian', 'Victim']  (2809ms + 3273ms)
  → Innocent→Antagonist (error) ... coarse=Innocent  fine=['Instigator', 'Victim']  (2281ms + 3759ms)
  → Protagonist (correct) ... coarse=Protagonist  fine=['Guardian', 'Instigator']  (4126ms + 4158ms)


In [ ]:
# =====================================
# S3. c9 — Occlusion saliency, one figure PER example (legible at page width)
#     Each figure is a 1x2 grid: coarse (left) + fine (right).
#     Saved as c9a/c9b/c9c so each fits a thesis page without shrinking text.
# =====================================
TOP_K = 8

def _draw_saliency_panel(ax, words, top_k, border_col=None, xlabel="Дял от общото влияние (%)"):
    """Draw one horizontal saliency bar panel; returns nothing."""
    if not words:
        ax.text(0.5, 0.5, "Няма фина роля", ha="center", va="center", fontsize=11)
        ax.axis("off")
        return
    total_abs = max(sum(abs(w["saliency"]) for w in words), 1e-12)
    ranked = sorted(words, key=lambda w: abs(w["saliency"]), reverse=True)[:top_k]
    ranked.sort(key=lambda w: w["saliency"], reverse=True)
    labels = [w["word"] for w in ranked]
    shares = [w["saliency"] / total_abs * 100.0 for w in ranked]
    colors = ["#2E7D32" if s > 0 else "#C62828" for s in shares]
    y = np.arange(len(ranked))
    bars = ax.barh(y, shares, color=colors, edgecolor="white", height=0.72)
    off = max(abs(s) for s in shares) * 0.03 if shares else 0.5
    for bar, s in zip(bars, shares):
        x = bar.get_width()
        ax.text(x + (off if x >= 0 else -off), bar.get_y() + bar.get_height()/2,
                f"{s:+.1f}%", va="center", ha="left" if x >= 0 else "right", fontsize=11)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=12)
    ax.invert_yaxis()
    ax.axvline(0, color="#888", linewidth=0.8)
    ax.grid(axis="x", alpha=0.25, linewidth=0.5)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    if border_col:
        ax.spines["left"].set_color(border_col); ax.spines["left"].set_linewidth(3)
    ax.set_xlabel(xlabel, fontsize=11)
    xmin = min(shares + [0.0]); xmax = max(shares + [0.0])
    span = (xmax - xmin) if xmax > xmin else max(abs(xmin), abs(xmax), 1.0)
    ax.set_xlim(xmin - span * 0.30, xmax + span * 0.30)

_c9_paths = ["c9a_baseline_saliency_antagonist.png",
             "c9b_baseline_saliency_innocent.png",
             "c9c_baseline_saliency_protagonist.png"]

for (ex, res, fname) in zip(SALIENCY_EXAMPLES, results_sal, _c9_paths):
    true_label = ex["true_coarse"]
    pred_label = res["pred_coarse"]
    correct    = pred_label == true_label
    mark       = "✓" if correct else "✗"
    border     = "#2E7D32" if correct else "#C62828"
    conf       = res["coarse_probs"].get(pred_label, 0.0)

    fig, (ax_c, ax_f) = plt.subplots(1, 2, figsize=(13, 4.6))
    _draw_saliency_panel(ax_c, res["words_coarse"], TOP_K, border_col=border)
    ax_c.set_title(f"Грубо ниво → {pred_label} ({conf:.0%}) {mark}   [истинско: {true_label}]",
                   fontsize=12, fontweight="bold", loc="left")

    fine_role = res["pred_fine"]
    fine_lbl  = res["top_fine_lbl"] or "—"
    _draw_saliency_panel(ax_f, res["words_fine"], TOP_K,
                         xlabel=f"Дял от влиянието върху {fine_lbl} (%)")
    ax_f.set_title(f"Фино ниво → {', '.join(fine_role) if fine_role else '—'}",
                   fontsize=12, fontweight="bold", loc="left")

    fig.suptitle(ex["label"], fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    out_path = os.path.join(BASELINE_DIAGRAMS_DIR, fname)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")


In [ ]:
# =====================================
# S4. c10 — Method comparison: occlusion vs gradient × embedding
#     Uses the first example (Antagonist) to compare both methods side by side.
#     Saves c10_baseline_saliency_method_comparison.png
# =====================================
ex_cmp  = SALIENCY_EXAMPLES[0]   # Antagonist correct example
res_cmp = results_sal[0]

mention = ex_cmp["mention"]
text    = ex_cmp["text"]
marked  = text.replace(mention, f"{ENTITY_START_TOKEN} {mention} {ENTITY_END_TOKEN}")
enc_cmp = _tokenize_marked(marked, tokenizer)
inp_cmp = enc_cmp["input_ids"].to(dev_str)
am_cmp  = enc_cmp["attention_mask"].to(dev_str)
target_id = label2id[res_cmp["pred_coarse"]]

# ── occlusion (already computed in S2) ───────────────────────────────────────
words_occ = res_cmp["words_coarse"]

# ── gradient × embedding ──────────────────────────────────────────────────────
print("Computing gradient × embedding saliency...", end=" ", flush=True)
t0 = time.perf_counter()
sal_grad = compute_grad_x_emb_saliency_baseline(
    inp_cmp, am_cmp, model, target_id, tokenizer, task="coarse", dev=dev_str
)
elapsed_g = (time.perf_counter() - t0) * 1000
print(f"{elapsed_g:.0f}ms")

if sal_grad is None:
    print("Gradient method failed — skipping comparison plot.")
else:
    words_grad = aggregate_words(sal_grad, enc_cmp, marked)

    # ── rank comparison table ─────────────────────────────────────────────────
    TOP_CMP = 10
    total_occ  = max(sum(abs(w["saliency"]) for w in words_occ),  1e-12)
    total_grad = max(sum(abs(w["saliency"]) for w in words_grad), 1e-12)

    ranked_occ  = sorted(words_occ,  key=lambda w: abs(w["saliency"]), reverse=True)[:TOP_CMP]
    ranked_grad = sorted(words_grad, key=lambda w: abs(w["saliency"]), reverse=True)[:TOP_CMP]

    top_occ_words  = [w["word"] for w in ranked_occ]
    top_grad_words = [w["word"] for w in ranked_grad]
    agreement = len(set(top_occ_words[:5]) & set(top_grad_words[:5]))
    print(f"\nTop-5 agreement between methods: {agreement}/5 words in common")
    print(f"Top-10 agreement: {len(set(top_occ_words) & set(top_grad_words))}/10 words in common")

    # ── figure: side-by-side bar charts + rank-diff scatter ───────────────────
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.6))
    fig.suptitle(
        f"Сравнение на методите — пример: {ex_cmp['label']}  (предсказано: {res_cmp['pred_coarse']})",
        fontsize=13, fontweight="bold"
    )

    def _draw_bar(ax, ranked, total, title_str):
        ranked.sort(key=lambda w: w["saliency"], reverse=True)
        labels = [w["word"] for w in ranked]
        shares = [w["saliency"] / total * 100.0 for w in ranked]
        colors = ["#2E7D32" if s > 0 else "#C62828" for s in shares]
        y = np.arange(len(ranked))
        bars = ax.barh(y, shares, color=colors, edgecolor="white", height=0.7)
        off = max(abs(s) for s in shares) * 0.03 if shares else 0.5
        for bar, s in zip(bars, shares):
            x = bar.get_width()
            ax.text(x + (off if x >= 0 else -off), bar.get_y() + bar.get_height() / 2,
                    f"{s:+.1f}%", va="center", ha="left" if x >= 0 else "right", fontsize=8)
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=11)
        ax.invert_yaxis()
        ax.axvline(0, color="#888", linewidth=0.8)
        ax.grid(axis="x", alpha=0.25, linewidth=0.5)
        for sp in ("top", "right"): ax.spines[sp].set_visible(False)
        ax.set_title(title_str, fontsize=12, fontweight="bold")
        ax.set_xlabel("Дял от общото влияние (%)", fontsize=10)
        if shares:
            xmin = min(shares + [0.0]); xmax = max(shares + [0.0])
            span = (xmax - xmin) if xmax > xmin else max(abs(xmin), abs(xmax), 1.0)
            ax.set_xlim(xmin - span * 0.28, xmax + span * 0.28)

    _draw_bar(axes[0], ranked_occ[:TOP_CMP],  total_occ,  "Оклузия (топ-10)")
    _draw_bar(axes[1], ranked_grad[:TOP_CMP], total_grad, "Градиент × Вграждане (топ-10)")

    # ── rank-correlation scatter (right panel) ────────────────────────────────
    ax_sc = axes[2]

    # Build a common vocabulary of all words with nonzero saliency in either method
    occ_dict  = {w["word"]: w["saliency"] / total_occ  * 100 for w in words_occ  if w["saliency"] != 0}
    grad_dict = {w["word"]: w["saliency"] / total_grad * 100 for w in words_grad if w["saliency"] != 0}
    common    = set(occ_dict) & set(grad_dict)

    if common:
        xs = [occ_dict[w]  for w in common]
        ys = [grad_dict[w] for w in common]
        pt_colors = ["#2E7D32" if x > 0 and y > 0 else
                     "#C62828" if x < 0 and y < 0 else
                     "#FF9800" for x, y in zip(xs, ys)]
        ax_sc.scatter(xs, ys, c=pt_colors, alpha=0.7, s=40, edgecolors="white", linewidths=0.5)

        # Label the top-5 by absolute occlusion
        top5 = sorted(common, key=lambda w: abs(occ_dict[w]), reverse=True)[:5]
        for w in top5:
            ax_sc.annotate(w, (occ_dict[w], grad_dict[w]),
                           textcoords="offset points", xytext=(4, 4), fontsize=7)

        lim = max(max(abs(x) for x in xs), max(abs(y) for y in ys)) * 1.15
        ax_sc.set_xlim(-lim, lim); ax_sc.set_ylim(-lim, lim)
        ax_sc.axhline(0, color="#aaa", linewidth=0.6, linestyle="--")
        ax_sc.axvline(0, color="#aaa", linewidth=0.6, linestyle="--")
        # Diagonal guide: perfect agreement
        ax_sc.plot([-lim, lim], [-lim, lim], color="#888", linewidth=0.8,
                   linestyle=":", label="пълно съгласие")
        ax_sc.legend(fontsize=8)

    ax_sc.set_xlabel("Оклузия, дял (%)", fontsize=10)
    ax_sc.set_ylabel("Градиент×Вгр., дял (%)", fontsize=10)
    ax_sc.set_title("Съгласуваност между методите\n(зелено/червено = съвпадащ знак, оранжево = разминаване)",
                    fontsize=11, fontweight="bold")
    ax_sc.grid(alpha=0.2)
    for sp in ("top", "right"): ax_sc.spines[sp].set_visible(False)

    plt.tight_layout()
    c10_path = os.path.join(BASELINE_DIAGRAMS_DIR, "c10_baseline_saliency_method_comparison.png")
    fig.savefig(c10_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {c10_path}")

    # ── summary printout ──────────────────────────────────────────────────────
    print(f"\n{'Word':<22} {'Occ share':>10} {'Grad share':>12} {'Sign match':>12}")
    print("-" * 58)
    for w in sorted(common, key=lambda w: abs(occ_dict[w]), reverse=True)[:15]:
        o, g = occ_dict[w], grad_dict[w]
        match = "✓" if (o > 0) == (g > 0) else "✗"
        print(f"  {w:<20} {o:>+9.1f}%  {g:>+9.1f}%  {match:>10}")
